In [0]:

from pyspark.sql import functions as F

# ==========================================
# E-COMMERCE DATA ENGINEERING PROJECT
# GOLD LAYER - BUSINESS ANALYTICS
# ==========================================

SCHEMA = "workspace.ecommerce"

print("STARTING GOLD LAYER")

# 1. LOAD SILVER TABLES
orders = spark.table(f"{SCHEMA}.silver_orders")
customers = spark.table(f"{SCHEMA}.silver_customers")
items = spark.table(f"{SCHEMA}.silver_order_items")
payments = spark.table(f"{SCHEMA}.silver_order_payments")
reviews = spark.table(f"{SCHEMA}.silver_order_reviews")
products = spark.table(f"{SCHEMA}.silver_products")
sellers = spark.table(f"{SCHEMA}.silver_sellers")
categories = spark.table(f"{SCHEMA}.silver_category_translation")

# 2. REMOVE DUPLICATE METADATA COLUMNS
metadata = [
    "_ingested_at",
    "_source_file",
    "_silver_processed_at"
]

def clean_metadata(df):
    return df.drop(*[c for c in metadata if c in df.columns])

orders = clean_metadata(orders)
customers = clean_metadata(customers)
items = clean_metadata(items)
payments = clean_metadata(payments)
reviews = clean_metadata(reviews)
products = clean_metadata(products)
sellers = clean_metadata(sellers)
categories = clean_metadata(categories)

print("Metadata cleaned successfully")

# 3. GOLD SALES TABLE
item_metrics = items.groupBy("order_id").agg(
    F.count("*").alias("total_items"),
    F.sum("price").alias("total_product_value"),
    F.sum("freight_value").alias("total_freight")
)

payment_metrics = payments.groupBy("order_id").agg(
    F.sum("payment_value").alias("total_payment"),
    F.count("*").alias("payment_count"),
    F.max("payment_installments").alias("max_installments")
)

review_metrics = reviews.groupBy("order_id").agg(
    F.avg("review_score").alias("average_review_score"),
    F.count("*").alias("review_count")
)

gold_sales = (
    orders
    .join(item_metrics, "order_id", "left")
    .join(payment_metrics, "order_id", "left")
    .join(review_metrics, "order_id", "left")
    .withColumn(
        "total_product_value",
        F.coalesce(F.col("total_product_value"), F.lit(0))
    )
    .withColumn(
        "total_freight",
        F.coalesce(F.col("total_freight"), F.lit(0))
    )
    .withColumn(
        "total_payment",
        F.coalesce(F.col("total_payment"), F.lit(0))
    )
    .withColumn(
        "total_order_value",
        F.col("total_product_value") + F.col("total_freight")
    )
    .withColumn(
        "purchase_year",
        F.year("order_purchase_timestamp")
    )
    .withColumn(
        "purchase_month",
        F.month("order_purchase_timestamp")
    )
    .withColumn(
        "purchase_date",
        F.to_date("order_purchase_timestamp")
    )
    .withColumn(
        "delivery_days",
        F.datediff(
            F.to_date("order_delivered_customer_date"),
            F.to_date("order_purchase_timestamp")
        )
    )
    .withColumn(
        "is_delivered",
        F.when(F.col("order_status") == "delivered", 1).otherwise(0)
    )
    .withColumn(
        "is_late",
        F.when(
            F.col("order_delivered_customer_date").isNotNull() &
            (F.col("order_delivered_customer_date") >
             F.col("order_estimated_delivery_date")),
            1
        ).otherwise(0)
    )
)

gold_sales.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{SCHEMA}.gold_sales")

print("SUCCESS: gold_sales")

# 4. GOLD CUSTOMERS
customer_orders = orders.groupBy("customer_id").agg(
    F.countDistinct("order_id").alias("total_orders"),
    F.min("order_purchase_timestamp").alias("first_order_date"),
    F.max("order_purchase_timestamp").alias("last_order_date")
)

customer_revenue = gold_sales.groupBy("customer_id").agg(
    F.sum("total_order_value").alias("customer_revenue"),
    F.avg("total_order_value").alias("average_order_value")
)

gold_customers = (
    customers
    .join(customer_orders, "customer_id", "left")
    .join(customer_revenue, "customer_id", "left")
    .fillna({
        "total_orders": 0,
        "customer_revenue": 0,
        "average_order_value": 0
    })
    .withColumn(
        "customer_type",
        F.when(F.col("total_orders") > 1, "Repeat Customer")
        .when(F.col("total_orders") == 1, "One-Time Customer")
        .otherwise("No Orders")
    )
)

gold_customers.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{SCHEMA}.gold_customers")

print("SUCCESS: gold_customers")

# 5. GOLD PRODUCTS
product_metrics = items.groupBy("product_id").agg(
    F.count("*").alias("units_sold"),
    F.countDistinct("order_id").alias("orders_containing_product"),
    F.sum("price").alias("product_revenue"),
    F.avg("price").alias("average_selling_price")
)

gold_products = (
    products
    .join(categories, "product_category_name", "left")
    .join(product_metrics, "product_id", "left")
    .fillna({
        "units_sold": 0,
        "orders_containing_product": 0,
        "product_revenue": 0
    })
    .withColumn(
        "product_category",
        F.coalesce(
            F.col("product_category_name_english"),
            F.col("product_category_name"),
            F.lit("Unknown")
        )
    )
)

gold_products.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{SCHEMA}.gold_products")

print("SUCCESS: gold_products")

# 6. GOLD PAYMENTS
gold_payments = payments.groupBy("payment_type").agg(
    F.count("*").alias("payment_transactions"),
    F.sum("payment_value").alias("total_payment_value"),
    F.avg("payment_value").alias("average_payment_value"),
    F.avg("payment_installments").alias("average_installments")
)

gold_payments.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{SCHEMA}.gold_payments")

print("SUCCESS: gold_payments")

# 7. GOLD DELIVERIES
gold_deliveries = (
    orders
    .filter(F.col("order_delivered_customer_date").isNotNull())
    .withColumn(
        "delivery_days",
        F.datediff(
            F.to_date("order_delivered_customer_date"),
            F.to_date("order_purchase_timestamp")
        )
    )
    .withColumn(
        "is_late",
        F.when(
            F.col("order_delivered_customer_date") >
            F.col("order_estimated_delivery_date"),
            1
        ).otherwise(0)
    )
    .withColumn(
        "delivery_status",
        F.when(F.col("is_late") == 1, "Late").otherwise("On Time")
    )
    .select(
        "order_id",
        "customer_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_days",
        "is_late",
        "delivery_status"
    )
)

gold_deliveries.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{SCHEMA}.gold_deliveries")

print("SUCCESS: gold_deliveries")

# 8. GOLD REVIEWS
gold_reviews = reviews.groupBy("review_score").agg(
    F.count("*").alias("total_reviews"),
    F.countDistinct("order_id").alias("orders_reviewed")
)

gold_reviews.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{SCHEMA}.gold_reviews")

print("SUCCESS: gold_reviews")

# 9. FINAL VALIDATION
gold_tables = [
    "gold_sales",
    "gold_customers",
    "gold_products",
    "gold_payments",
    "gold_deliveries",
    "gold_reviews"
]

results = []

for table in gold_tables:
    full_name = f"{SCHEMA}.{table}"
    count = spark.table(full_name).count()
    results.append((table, count))
    print(f"VERIFIED: {full_name} | Rows: {count}")

summary_df = spark.createDataFrame(
    results,
    ["gold_table", "row_count"]
)

print("====================================")
print("GOLD LAYER COMPLETED SUCCESSFULLY")
print("====================================")

display(summary_df)


STARTING GOLD LAYER
Metadata cleaned successfully
SUCCESS: gold_sales
SUCCESS: gold_customers
SUCCESS: gold_products
SUCCESS: gold_payments
SUCCESS: gold_deliveries
SUCCESS: gold_reviews
VERIFIED: workspace.ecommerce.gold_sales | Rows: 99441
VERIFIED: workspace.ecommerce.gold_customers | Rows: 99441
VERIFIED: workspace.ecommerce.gold_products | Rows: 32951
VERIFIED: workspace.ecommerce.gold_payments | Rows: 5
VERIFIED: workspace.ecommerce.gold_deliveries | Rows: 96476
VERIFIED: workspace.ecommerce.gold_reviews | Rows: 7
GOLD LAYER COMPLETED SUCCESSFULLY


gold_table,row_count
gold_sales,99441
gold_customers,99441
gold_products,32951
gold_payments,5
gold_deliveries,96476
gold_reviews,7
